# FreeSolv validation example

**Notebook type:** experiment. This optional validation example runs independently of
the main SGLT2 notebooks. It uses the complete
FreeSolv v0.52 hydration dataset, using experimental free energies in **kcal/mol**.
More negative values mean more favorable hydration. No pIC50 transform or sign reversal is applied.

Run in the CheMeleon extraction environment (Python >=3.11, Chemprop 2.2.0).
The same additive GNN implementation, four loss terms, and Ridge fitter are used.
This lightweight example uses a two-layer GCN with 32 hidden units, rather than
the selected SGLT2 GATv2 configuration. FreeSolv has
one dataset-wide MMP group, rather than the SGLT2 document groups. Its fast 2D atom
descriptors are deliberately different from the paper's costly CREST/AIMNet/OpenFF
features, and are not CheMeleon inputs. This record tests execution and measured
constraint behavior; it is not a benchmark or a toxicity-classification validation.

In [ ]:
LOCAL_PROJECT_ROOT = "."
EPOCHS = 100
SEED = 42
RUN_MODE = "train"
RUN_NAME = "freesolv_transfer"
CHECKPOINT_PATH = "checkpoints/chemeleon_mp.pt"

## 1. Runtime and public API

In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display

# Use the declared repository root in both Colab and local execution.
try:
    import google.colab
except ModuleNotFoundError as exc:
    if exc.name not in {"google", "google.colab"}:
        raise
    IS_COLAB = False
else:
    IS_COLAB = True
ROOT = Path(os.environ.get("XAI_CHEM_ROOT", LOCAL_PROJECT_ROOT)).expanduser().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print({"runtime": "colab" if IS_COLAB else "local", "root": str(ROOT), "python": sys.executable})

# Numerical tables, chemistry figures, and explicit CPU execution.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)
%matplotlib inline
from xai_chem.transfer import load_freesolv_snapshot, build_regression_mmp, rdkit_atom_descriptors, RDKit_DESCRIPTORS
from xai_chem.graphs import build_graphs
from xai_chem.representations import load_molecule_inputs_with_provenance, CheMeleonSingletonProvider, build_or_reuse_cache
from xai_chem.dataset import DataConfig, CacheBackedGraphDataset
from xai_chem.config import TrainingConfig
from xai_chem.training import run_fold_training
from xai_chem.results import load_completed_fold_results, check_additivity, assert_completed_result_matches_config


## 2. Inspect the experimental endpoint and fix a random split

In [ ]:
run_root = ROOT / "test_sample/output" / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
frame = load_freesolv_snapshot(ROOT / "test_sample/input_data/freesolv/database.txt")
dataset_path = run_root / "freesolv.csv"
frame.to_csv(dataset_path, index=False)
permutation = np.random.default_rng(SEED).permutation(len(frame))
fold_ids = np.empty(len(frame), dtype=int)
fold_ids[permutation] = np.arange(len(frame)) % 5
fold_table = pd.DataFrame({"index": frame["index"], "fold": fold_ids})
fold_path = run_root / "folds.csv"
fold_table.to_csv(fold_path, index=False)
print({"molecules": len(frame), "target_units": "kcal/mol", "validation_fold": 0,
       "split_seed": SEED, "fold_counts": fold_table["fold"].value_counts().sort_index().to_dict()})

## 3. Build heavy-atom topology with the explicit solvation target

In [ ]:
graph_root = run_root / "graphs"
RUN_GRAPH_BUILD = not graph_root.exists()
print({"run_graph_build": RUN_GRAPH_BUILD})
if RUN_GRAPH_BUILD:
    build_graphs(dataset_path, graph_root, target_column="hydration_kcal_mol")
loaded = load_molecule_inputs_with_provenance(dataset_path, graph_root, dataset_id="FreeSolv_v0.52")
print({"included": len(loaded.molecules), "excluded": len(loaded.exclusions),
       "atoms": sum(len(mol.atomic_numbers) for mol in loaded.molecules)})

## 4. Extract the actual pretrained atom representations

Each molecule is encoded alone. The same raw CheMeleon weights are used for both training conditions.

In [ ]:
provider = CheMeleonSingletonProvider(checkpoint_path=ROOT / CHECKPOINT_PATH, device="cpu")
bundle = build_or_reuse_cache(run_root / "representations", provider, loaded.molecules,
                             input_provenance=loaded.input_provenance, provider_scoped_root=True)
print({"shape": bundle.values.shape, "extraction_mode": bundle.metadata["extraction_mode"],
       "provider": bundle.metadata["provider"]["provider"]})

## 5. Construct single-cut MMPs and correct chemical core correspondence

The original cut SMARTS, selection thresholds, and automorphism-aware core mapping are reused. FreeSolv MMPs are grouped at dataset level; no source document is inferred.
The inherited pair filters require Morgan similarity ≥ 0.1 and |Δtarget| ≥ 0.1
in the stated native target units. Training uses only pairs whose two endpoints
are training molecules; validation pairs are used only for audits/reconstruction.


In [ ]:
mmp_path = run_root / "mmp.csv"
if not mmp_path.exists():
    mmp = build_regression_mmp(frame, graph_root, mmp_path)
else:
    mmp = pd.read_csv(mmp_path)
train_rows = set(np.flatnonzero(fold_ids != 0))
validation_rows = set(np.flatnonzero(fold_ids == 0))
train_pairs = mmp["i_id"].isin(train_rows) & mmp["j_id"].isin(train_rows)
validation_pairs = mmp["i_id"].isin(validation_rows) & mmp["j_id"].isin(validation_rows)
print({"total_pairs": len(mmp), "training_pairs": int(train_pairs.sum()),
       "validation_pairs": int(validation_pairs.sum()), "dy_units": "kcal/mol"})

## 6. Fix matched U/R training conditions before optimization

Both conditions use the same architecture, initialization seed, target scaling,
training rows, optimizer, and 100-epoch budget. The final epoch is retained;
validation labels do not select the checkpoint. U sets both auxiliary outer weights
to zero. R activates the original four losses. Frequent-fragment positivity is
a hypothesis about the allocation in standardized target space, not a claim that
hydration contributions must be physically positive.

In [ ]:
data_config = DataConfig(dataset_csv=dataset_path, graph_root=graph_root, fold_csv=fold_path,
                         representation_cache=run_root / "representations", target_column="hydration_kcal_mol")
raw_dataset = CacheBackedGraphDataset(data_config, validate_all=True)
common = dict(data=data_config, mmp_csv=mmp_path, fragment_cache=None, device="cpu",
              epochs=EPOCHS, seed=SEED, validation_fold=0, expected_train_rows=len(train_rows),
              expected_retained_mmp_pairs=int(train_pairs.sum()), expected_mmp_source_pairs=len(mmp),
              expected_mmp_symmetry_ambiguous_pairs=int((mmp["core_pair_automorphism_candidate_count"] > 1).sum()),
              expected_mmp_max_automorphism_candidates=int(mmp["core_pair_automorphism_candidate_count"].max()),
              backbone_name="gcn", hidden_dim=32, num_layers=2, use_edge_features=False,
              heads=None, dropout=0.0, backbone_normalization="layernorm", readout_hidden_dim=32,
              readout_num_layers=2, output_head_init_std=0.001, optimization_mode="full_objective",
              checkpoint_selection="fixed_final_epoch", early_stopping_patience=None,
              learning_rate=0.002, weight_decay=0.0001, mmp_pair_batch_size=2048,
              supervised_batch_size=128, mmp_core_weight=3.0, mmp_difference_weight=1.0,
              fragment_positive_weight=1.0, fragment_consistency_weight=1.0)
configs = {label: TrainingConfig(output_dir=run_root / label, mmp_weight=pair_weight,
                                fragment_frequency_weight=fragment_weight, **common)
           for label, pair_weight, fragment_weight in [("U", 0.0, 0.0), ("R", 5.0, 1.0)]}
print({"backbone": common["backbone_name"], "hidden_dim": common["hidden_dim"],
       "epochs": EPOCHS, "seed": SEED, "checkpoint_selection": common["checkpoint_selection"],
       "mmp_outer_weights": {label: config.mmp_weight for label, config in configs.items()},
       "fragment_outer_weights": {label: config.fragment_frequency_weight for label, config in configs.items()}})


## 7. Train both models through the published training engine

In [ ]:
completed = {}
def print_progress(row):
    if int(row["epoch"]) == 1 or int(row["epoch"]) % 20 == 0:
        print({key: row[key] for key in ("epoch", "optimization_total", "training_rmse_raw", "validation_rmse_raw")}, flush=True)

for label, config in configs.items():
    print(f"Condition {label}", flush=True)
    if RUN_MODE == "train":
        run_fold_training(config, raw_dataset=raw_dataset, epoch_observer=print_progress)
    elif RUN_MODE != "reuse":
        raise ValueError("RUN_MODE must be train or reuse")
    completed[label] = load_completed_fold_results(config.output_dir)
    assert_completed_result_matches_config(completed[label], config)
    print({"condition": label, "epochs_saved": len(completed[label].history)})


## 8. Measure all four constraints on matched atom and pair populations

In [ ]:
from xai_chem.transfer import measure_ac_constraints

constraint_rows = []
for label, result in completed.items():
    for split in ("train", "validation"):
        constraint_rows.append({"condition": label, **measure_ac_constraints(result, raw_dataset, mmp_path, split=split)})
constraints = pd.DataFrame(constraint_rows)
display(constraints)
constraints.to_csv(run_root / "ac_constraints.csv", index=False)
train_constraints = constraints.query("split == 'train'").set_index("condition")
ratios = train_constraints.loc["R", ["core", "difference", "positive", "consistency"]] / train_constraints.loc["U", ["core", "difference", "positive", "consistency"]]
print({"training_constraint_ratios_R_div_U": ratios.astype(float).round(4).to_dict()})


The ratios compare raw loss components evaluated with identical definitions. Values below one show that the regularized model better satisfies that measured constraint. Validation is reported separately and is not used to tune these weights.

## 9. Inspect predictive fit, additivity, and learning curves

Use the frozen project Y–Y style (prediction on x; experiment on y) and
raw/light plus smoothed/dark scaled-MSE curves. The selected checkpoint is explicit.


In [ ]:
TARGET_LABEL = "hydration ΔG (kcal/mol)"
TARGET_UNITS = "kcal/mol"
from xai_chem.plotting import (
    plot_train_validation_yy, plot_prediction_loss_curves, save_publication_figure,
)
# Compare U/R within each figure type; keep the established train/validation style.
all_predictions = pd.concat([result.predictions for result in completed.values()])
span = all_predictions[["target_raw", "prediction_raw"]].to_numpy()
padding = 0.03 * max(float(span.max() - span.min()), 1.0)
limits = (float(span.min() - padding), float(span.max() + padding))
figure, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)
summary_rows, additivity_passed = [], {}
for axis, label in zip(axes, ("U", "R")):
    result = completed[label]
    _, scores = plot_train_validation_yy(result, run_name=label, limits=limits,
                                         target_label=TARGET_LABEL, axis=axis)
    axis.set_title(f"{label} · fold 0")
    summary_rows.append(scores.assign(condition=label))
    audit = check_additivity(result.predictions, result.atom_contributions)
    additivity_passed[label] = bool(audit["scaled_pass"].all() and audit["raw_pass"].all())
    if not additivity_passed[label]:
        raise ValueError(f"Additivity failed for {label}")
save_publication_figure(figure, run_root / "vis" / "UR_yy.png")
display(figure); plt.close(figure)
summary = pd.concat(summary_rows, ignore_index=True)
display(summary[["condition", "split", "n", "r2_score_raw", "rmse_raw"]].round(4))
summary.to_csv(run_root / "prediction_metrics.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True, sharey=True)
for axis, label in zip(axes, ("U", "R")):
    plot_prediction_loss_curves(completed[label], axis=axis)
    axis.set_title(f"{label} · prediction loss · fold 0")
figure.tight_layout()
save_publication_figure(figure, run_root / "vis" / "UR_learning.png")
display(figure); plt.close(figure)
print({"additivity_passed": additivity_passed})


## 10. Three large experimental contrasts; R-only reconstruction

Select the three largest |experimental B−A| gaps among validation/validation pairs,
with distinct transformations and pair ID as the tie-breaker. This is a deliberately
selected illustration, not a representative performance sample. Fit the unchanged
seven RDKit descriptors and Ridge alpha=1 to R changed-region ACs only.


In [ ]:
from xai_chem.reconstruction import fit_pair
# Select distinct transformations by observed target gap, never by AC/fidelity.
# These deliberately strong contrasts illustrate the method, not average performance.
selected_pairs = (mmp.loc[validation_pairs]
    .assign(abs_target_difference=lambda rows: (rows["target_j"] - rows["target_i"]).abs())
    .sort_values(["abs_target_difference", "pair_id"], ascending=[False, True], kind="stable")
    .drop_duplicates("mmp_transform").head(3))
selected_pairs.to_csv(run_root / "selected_mmp_examples.csv", index=False)
descriptors = rdkit_atom_descriptors(frame)
teacher = completed["R"].atom_contributions.rename(columns={
    "source_row_index": "graph_id", "smiles_atom_index": "atom_idx",
    "atom_contribution_raw": "atom_contribution",
})
atom_table = descriptors.merge(teacher[["graph_id", "atom_idx", "atom_contribution"]],
                               on=["graph_id", "atom_idx"], validate="one_to_one")
fitted_pairs, fits = {}, []
for pair in selected_pairs.itertuples(index=False):
    context = f"R_pair_{pair.pair_id}"
    pieces = []
    for graph_id, indices in [(pair.i_id, json.loads(pair.diff_i_json)),
                              (pair.j_id, json.loads(pair.diff_j_json))]:
        pieces.append(atom_table[atom_table["graph_id"].eq(graph_id)
                     & atom_table["atom_idx"].isin(indices)].assign(context_id=context))
    inputs = pd.concat(pieces, ignore_index=True)
    key = pd.Series({"context_id": context, "pair_id": pair.pair_id,
                     "molecule_a": frame.iloc[pair.i_id]["cid"],
                     "molecule_b": frame.iloc[pair.j_id]["cid"]})
    fitted = fit_pair(key, inputs, alpha=1.0, features=RDKit_DESCRIPTORS)
    fitted.atoms.to_csv(run_root / f"R_pair_{pair.pair_id}_atom_reconstruction.csv", index=False)
    fitted.concept_totals.to_csv(run_root / f"R_pair_{pair.pair_id}_descriptor_effects.csv", index=False)
    fits.append(fitted.metrics.assign(condition="R"))
    fitted_pairs[pair.pair_id] = fitted
reconstruction = pd.concat(fits, ignore_index=True)
reconstruction.to_csv(run_root / "descriptor_reconstruction.csv", index=False)
# One compact table replaces repeated fit, coefficient, and atom tables.
pair_summary = selected_pairs[["pair_id", "target_i", "target_j", "abs_target_difference"]].merge(
    reconstruction.loc[reconstruction["level"].eq("A+B diff-region atoms"), ["pair_id", "n", "r2"]],
    on="pair_id", validate="one_to_one")
display(pair_summary.rename(columns={"r2": "R_in_sample_atom_R2"}).round(4))


## 11. Matched MMP maps: U/R rows, A/B columns

Three 2×2 panels use one raw AC scale and no atom indices. Titles retain predictions,
observations, errors and target units. Detailed atom values, affine baselines and
saturation audits are saved to CSV instead of repeated inline tables.


In [ ]:
from xai_chem.plotting import plot_mmp_condition_comparison
# A single raw scale covers both conditions and every selected molecule.
selected_graph_ids = set(selected_pairs["i_id"]) | set(selected_pairs["j_id"])
shared_limit = max(float(result.atom_contributions.loc[
    result.atom_contributions["source_row_index"].isin(selected_graph_ids),
    "atom_contribution_raw"].abs().max()) for result in completed.values())
print({"shared_raw_ac_symmetric_limit": shared_limit, "raw_ac_units": TARGET_UNITS,
       "atom_indices_displayed": False})
for pair in selected_pairs.itertuples(index=False):
    figure, molecule_audit, atom_audit = plot_mmp_condition_comparison(
        completed, source_rows=[pair.i_id, pair.j_id], symmetric_limit=shared_limit,
        unit_label=TARGET_UNITS,
    )
    figure.suptitle(f"Pair {pair.pair_id} · |experimental B−A| = {pair.abs_target_difference:.3f} {TARGET_UNITS}")
    save_publication_figure(figure, run_root / "vis" / f"pair_{pair.pair_id}_UR_2x2.png")
    molecule_audit.to_csv(run_root / f"pair_{pair.pair_id}_map_audit.csv", index=False)
    atom_audit.to_csv(run_root / f"pair_{pair.pair_id}_raw_atom_values.csv", index=False)
    display(figure); plt.close(figure)


The raw maps describe model evidence; a shared scale makes U/R directly comparable.
Large observed gaps do not guarantee accurate predictions or chemically correct ACs.


## R-only reconstruction fidelity

These fits are in-sample on changed-region atoms. They do not measure unseen-atom prediction or causal attribution.

In [ ]:
from xai_chem.plotting import plot_reconstruction_yy
# R-only surrogate fidelity: U/R comparison is reserved for the MMP maps.
figure, axes = plt.subplots(1, len(selected_pairs), figsize=(6 * len(selected_pairs), 5.5), squeeze=False)
for axis, pair in zip(axes[0], selected_pairs.itertuples(index=False)):
    fitted = fitted_pairs[pair.pair_id]
    plot_reconstruction_yy(fitted.atoms["atom_contribution"], fitted.atoms["prediction"],
        unit_label=f"AC ({TARGET_UNITS})", title=f"R · pair {pair.pair_id}", axis=axis)
figure.tight_layout()
save_publication_figure(figure, run_root / "vis" / "R_reconstruction_yy.png")
display(figure); plt.close(figure)


## Descriptor interpretation of one selected MMP

The first selected pair is used without inspecting fidelity. Rank descriptors by the absolute signed contribution to reconstructed changed-region B−A; the shared atom bias is reported separately. Correlated descriptors can share evidence, so ranks are descriptive.

In [ ]:
# Use the first selected MMP as a declared example; do not select by fit quality.
example_pair = next(selected_pairs.itertuples(index=False))
fitted = fitted_pairs[example_pair.pair_id]
effects = fitted.concept_totals.drop_duplicates("concept").set_index("concept")["B_minus_A"]
descriptor_effects = effects.loc[list(RDKit_DESCRIPTORS)]
# Rank pair-difference effects, not coefficients or arbitrary display contrast.
ranking = descriptor_effects.rename("effect_B_minus_A").rename_axis("descriptor").reset_index()
ranking["absolute_effect"] = ranking["effect_B_minus_A"].abs()
ranking = ranking.sort_values(["absolute_effect", "descriptor"], ascending=[False, True], kind="stable")
top_descriptors = ranking.head(2)["descriptor"].tolist()
ranking.to_csv(run_root / f"R_pair_{example_pair.pair_id}_descriptor_ranking.csv", index=False)
figure, axis = plt.subplots(figsize=(9.5, 4.5))
axis.barh(ranking["descriptor"], ranking["effect_B_minus_A"], color="0.5")
axis.invert_yaxis(); axis.axvline(0, color="black", linewidth=1)
axis.set(xlabel=f"Signed descriptor effect on fitted B−A ({TARGET_UNITS})",
         title=f"R · pair {example_pair.pair_id} · changed-region reconstruction")
axis.grid(axis="x", alpha=0.3)
figure.tight_layout()
save_publication_figure(figure, run_root / "vis" / "R_descriptor_effects.png")
display(figure); plt.close(figure)
print({"top_two_descriptors": top_descriptors,
       "shared_atom_bias_B_minus_A": float(effects.loc["shared atom bias"]),
       "reconstructed_changed_region_B_minus_A": float(effects.sum())})


### Atom maps: R AC, highest-ranked descriptor, second-ranked descriptor

Rows are A/B molecules; columns are AC and two raw descriptors. Descriptor halos cover the fitted changed-region atoms; unhighlighted core atoms were not fitted. Each descriptor uses its dataset-wide raw range and the established gist_rainbow palette. Color order is not a perceptually uniform magnitude scale; read the colorbars. These fast 2D RDKit descriptors differ from the quantum/conformer descriptors in the SGLT2 paper.

In [ ]:
from xai_chem.plotting import plot_pair_descriptor_maps
# Show raw descriptor values on the same changed atoms that entered the surrogate.
figure, descriptor_map_audit = plot_pair_descriptor_maps(
    completed["R"], descriptors, fitted.atoms,
    source_rows=[example_pair.i_id, example_pair.j_id], top_descriptors=top_descriptors,
    symmetric_limit=shared_limit, unit_label=TARGET_UNITS,
)
figure.suptitle(f"R · pair {example_pair.pair_id} · AC and top two descriptors")
save_publication_figure(figure, run_root / "vis" / "R_AC_top_descriptors_2x3.png")
descriptor_map_audit.to_csv(run_root / "descriptor_map_atom_values.csv", index=False)
display(figure); plt.close(figure)


The maps locate the raw descriptor values associated with this pair-local surrogate explanation. They are not maps of causal effects or experimental atomic properties.

## 12. Acceptance record

In [ ]:
training_decreased = {
    label: bool(result.history.iloc[-1]["optimization_prediction_mse_scaled"] < result.history.iloc[0]["optimization_prediction_mse_scaled"])
    for label, result in completed.items()
}
record = {"dataset": "FreeSolv_v0.52", "molecules": len(frame), "target": "hydration_kcal_mol",
          "target_units": "kcal/mol", "fold": 0, "seed": SEED, "epochs": EPOCHS,
          "training_prediction_loss_decreased": training_decreased,
          "all_models_additive": all(additivity_passed.values()),
          "all_four_training_constraints_lower": bool((ratios < 1).all()),
          "training_constraint_ratios_R_div_U": {key: float(value) for key, value in ratios.items()},
          "descriptor_pairs": len(selected_pairs), "ridge_alpha": 1.0,
          "descriptor_contract": list(RDKit_DESCRIPTORS), "reconstruction_scope": "in_sample_diff_region_atoms"}
print({"dataset": record["dataset"], "training_prediction_loss_decreased": training_decreased,
       "all_models_additive": record["all_models_additive"], "selected_mmp_pairs": len(selected_pairs),
       "reconstruction_condition": "R", "atom_indices_displayed": False})
(run_root / "acceptance.json").write_text(json.dumps(record, indent=2, allow_nan=False) + "\n")
if not all(training_decreased.values()) or not record["all_four_training_constraints_lower"] or len(selected_pairs) != 3:
    raise ValueError("FreeSolv acceptance condition was not met; inspect the measured results")

The experiment establishes the measured training and constraint behavior only for the declared split and settings. A lower regularization penalty is not evidence of physically correct ACs. Ridge fidelity is in-sample reconstruction, not predictive accuracy on unseen atoms. Negative hydration energies are preserved, and recurrent-fragment positivity remains a model-design assumption that should be reconsidered for each endpoint.